# 🎬 Movie Recommendation System - Step-by-Step Notebook

This notebook walks through the complete ML pipeline for building a **Content-Based Movie Recommendation System** using the TMDB 5000 Movies Dataset.

**Author:** Abhishek  
**Project:** Codec Technology Internship  
**Tech Stack:** Python, Pandas, NumPy, Scikit-learn, NLTK

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer

print('All libraries loaded successfully!')

## Step 2: Load the Dataset

We use two CSV files from the TMDB 5000 Movies Dataset:
- `tmdb_5000_movies.csv` — contains movie metadata (genres, keywords, overview, etc.)
- `tmdb_5000_credits.csv` — contains cast and crew information

These files are downloaded automatically by `preprocess.py`. Here we load them from URLs for demonstration.

In [ ]:
# Load datasets from Hugging Face mirrors
movies = pd.read_csv('https://huggingface.co/datasets/Adpacci/tmdb-5000/resolve/main/tmdb_5000_movies.csv')
credits = pd.read_csv('https://huggingface.co/datasets/Adpacci/tmdb-5000/resolve/main/tmdb_5000_credits.csv')

print(f'Movies shape: {movies.shape}')
print(f'Credits shape: {credits.shape}')

In [ ]:
# Preview movies dataset
movies.head(3)

In [ ]:
# Preview credits dataset
credits.head(3)

## Step 3: Merge Datasets

We merge both datasets on the `title` column to combine metadata with cast/crew info.

In [ ]:
# Merge movies and credits on 'title'
df = movies.merge(credits, on='title')
print(f'Merged shape: {df.shape}')
df.columns.tolist()

## Step 4: Select Useful Features & Clean Missing Values

We keep only the columns needed for our recommendation engine:
- `id` — TMDB movie ID (used to fetch posters)
- `title` — Movie title
- `overview` — Plot summary
- `genres` — Movie genres
- `keywords` — Movie keywords/tags
- `cast` — Actors
- `crew` — Director information

In [ ]:
# Select relevant columns
df = df[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Check for missing values
print('Missing values before cleaning:')
print(df.isnull().sum())

# Drop rows with missing values
df.dropna(inplace=True)
print(f'\nShape after dropping nulls: {df.shape}')

## Step 5: Feature Extraction from JSON Columns

The `genres`, `keywords`, `cast`, and `crew` columns contain JSON-like strings (list of dictionaries). We need to extract meaningful text from these.

In [ ]:
# Let's examine what these columns look like
print('Sample genres value:')
print(df.iloc[0]['genres'])
print('\nSample cast value (first 200 chars):')
print(df.iloc[0]['cast'][:200])

In [ ]:
# Helper function: Extract names from list of dicts (for genres & keywords)
def convert(obj):
    """Extract list of names from JSON string."""
    names = []
    for item in ast.literal_eval(obj):
        names.append(item['name'])
    return names

# Helper function: Extract top 3 cast members
def convert_cast(obj):
    """Extract top 3 actor names."""
    names = []
    counter = 0
    for item in ast.literal_eval(obj):
        if counter < 3:
            names.append(item['name'])
            counter += 1
        else:
            break
    return names

# Helper function: Extract Director from crew
def fetch_director(obj):
    """Find the Director from the crew list."""
    for item in ast.literal_eval(obj):
        if item.get('job') == 'Director':
            return [item['name']]
    return []

print('Helper functions defined!')

In [ ]:
# Apply feature extraction
df['genres'] = df['genres'].apply(convert)
df['keywords'] = df['keywords'].apply(convert)
df['cast'] = df['cast'].apply(convert_cast)
df['director'] = df['crew'].apply(fetch_director)

# Preview extracted features for Avatar
print('Avatar features:')
print(f"  Genres: {df.iloc[0]['genres']}")
print(f"  Keywords: {df.iloc[0]['keywords']}")
print(f"  Cast: {df.iloc[0]['cast']}")
print(f"  Director: {df.iloc[0]['director']}")

## Step 6: Normalize Text Features

We remove spaces from multi-word names so they are treated as single tokens:
- `"Science Fiction"` → `"ScienceFiction"`
- `"Sam Worthington"` → `"SamWorthington"`

This ensures the vectorizer doesn't split entity names into separate words.

In [ ]:
# Collapse spaces in named entities
def collapse_spaces(L):
    return [name.replace(' ', '') for name in L]

df['genres'] = df['genres'].apply(collapse_spaces)
df['keywords'] = df['keywords'].apply(collapse_spaces)
df['cast'] = df['cast'].apply(collapse_spaces)
df['director'] = df['director'].apply(collapse_spaces)

# Split overview into word list
df['overview'] = df['overview'].apply(lambda x: x.split())

print('After normalization - Avatar:')
print(f"  Genres: {df.iloc[0]['genres']}")
print(f"  Cast: {df.iloc[0]['cast']}")

## Step 7: Create Unified Tags Column

Combine all text features into a single `tags` column. This is the core idea behind content-based filtering — we represent each movie as a "bag of words".

In [ ]:
# Combine all features into 'tags'
df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['director']

# Create the final processed DataFrame
processed = df[['id', 'title', 'genres', 'director', 'tags']].copy()
processed['tags'] = processed['tags'].apply(lambda x: ' '.join(x).lower())

print(f'Processed DataFrame shape: {processed.shape}')
print(f'\nSample tags for Avatar (first 300 chars):')
print(processed.iloc[0]['tags'][:300])

## Step 8: Apply Stemming (NLTK PorterStemmer)

Stemming reduces words to their root form. For example:
- `loving`, `loved`, `loves` → `love`
- `dancing`, `danced` → `danc`

This helps improve matching accuracy.

In [ ]:
# Apply Porter Stemmer
ps = PorterStemmer()

def stem_text(text):
    return ' '.join([ps.stem(word) for word in text.split()])

processed['tags'] = processed['tags'].apply(stem_text)

# Show stemmed tags for Avatar
print('Stemmed tags for Avatar (first 300 chars):')
print(processed.iloc[0]['tags'][:300])

## Step 9: Vectorize Using CountVectorizer (Bag of Words)

Convert text tags into numerical vectors using `CountVectorizer`. We limit vocabulary to the **top 5000 most frequent words** and exclude common English stop words.

In [ ]:
# Vectorize using CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(processed['tags']).toarray()

print(f'Vector matrix shape: {vectors.shape}')
print(f'Vocabulary size: {len(cv.vocabulary_)}')
print(f'\nSample vocabulary words: {list(cv.vocabulary_.keys())[:15]}')

## Step 10: Calculate Cosine Similarity

Cosine similarity measures the angle between two vectors. Movies with similar tag vectors will have higher cosine similarity scores (closer to 1.0).

In [ ]:
# Calculate cosine similarity matrix
similarity = cosine_similarity(vectors)

print(f'Similarity matrix shape: {similarity.shape}')
print(f'\nSimilarity of Avatar with itself: {similarity[0][0]:.4f}')
print(f'Similarity of Avatar with 2nd movie: {similarity[0][1]:.4f}')

## Step 11: Build the Recommendation Function

Given a movie title, find the top 10 most similar movies based on cosine similarity scores.

In [ ]:
def recommend(movie_title):
    """Recommend top 10 similar movies."""
    # Find the index of the selected movie
    movie_index = processed[processed['title'] == movie_title].index[0]
    
    # Get similarity scores for all movies
    distances = similarity[movie_index]
    
    # Sort by similarity (descending) and pick top 10 (excluding itself)
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:11]
    
    print(f'\n🎬 Top 10 movies similar to "{movie_title}":\n')
    print(f'{"Rank":<6}{"Title":<45}{"Similarity":<12}')
    print('-' * 63)
    for rank, (idx, score) in enumerate(movie_list, 1):
        print(f'{rank:<6}{processed.iloc[idx]["title"]:<45}{score:.4f}')

# Test with Avatar
recommend('Avatar')

In [ ]:
# Test with The Dark Knight
recommend('The Dark Knight')

In [ ]:
# Test with Interstellar
recommend('Interstellar')

## Step 12: Save Processed Data Using Pickle

We save the processed DataFrame and similarity matrix as `.pkl` files so the Streamlit app can load them instantly without re-computing.

In [ ]:
# Save movies DataFrame
pickle.dump(processed, open('../movies.pkl', 'wb'))
print('Saved movies.pkl')

# Save similarity matrix (cast to float32 to save space)
similarity_f32 = similarity.astype(np.float32)
pickle.dump(similarity_f32, open('../similarity.pkl', 'wb'))
print('Saved similarity.pkl')

print(f'\nMovies pkl: {processed.shape}')
print(f'Similarity pkl: {similarity_f32.shape}')
print('\n✅ All files saved successfully! You can now run: streamlit run app.py')

---

## Summary

| Step | Description | Technique |
|------|-------------|----------|
| 1 | Load Dataset | Pandas `read_csv()` |
| 2 | Merge Movies + Credits | `pd.merge()` on title |
| 3 | Select Features | `id`, `title`, `overview`, `genres`, `keywords`, `cast`, `crew` |
| 4 | Clean Missing Values | `dropna()` |
| 5 | Parse JSON Columns | `ast.literal_eval()` |
| 6 | Normalize Text | Remove spaces in entity names |
| 7 | Create Tags | Combine all features into one text column |
| 8 | Stemming | NLTK `PorterStemmer` |
| 9 | Vectorization | `CountVectorizer(max_features=5000)` |
| 10 | Cosine Similarity | `sklearn.metrics.pairwise.cosine_similarity` |
| 11 | Recommendation | Sort by similarity score |
| 12 | Save Models | `pickle.dump()` |

**Next Step:** Run `streamlit run app.py` to launch the web application!